# Wheeze Detection from Children's Breathing Sounds

**A working AI demo — no installation, runs in your browser.**

This notebook teaches a computer to listen to a short recording of a child's breathing and decide **wheeze** vs **no wheeze**. It uses **SPRSound**, a free public dataset of children's lung sounds that doctors have already labelled.

### How to run it
1. In the menu: **Runtime → Change runtime type → GPU** (optional, makes the CNN faster).
2. Menu: **Runtime → Run all**.
3. Wait a few minutes. Scroll down to see the accuracy results and charts.

Each grey code box runs by itself; you don't need to edit anything.

## Step 1 — Install the software and download the dataset

In [ ]:
!pip -q install librosa xgboost
import os
if not os.path.exists('SPRSound'):
    !git clone --depth 1 https://github.com/SJTU-YONGFU-RESEARCH-GRP/SPRSound.git
print('\nReady.')

## Step 2 — Settings (change nothing the first time)

In [ ]:
from pathlib import Path

WAV_DIR  = Path('SPRSound/BioCAS2022/train2022_wav')
JSON_DIR = Path('SPRSound/BioCAS2022/train2022_json')

SAMPLE_RATE   = 8000      # SPRSound is recorded at 8 kHz
CLIP_SECONDS  = 2.0       # every clip is cut/padded to this length
BANDPASS_LOW  = 80        # keep lung-sound frequencies (~80..2000 Hz)
BANDPASS_HIGH = 2000
N_MFCC = 40
N_MELS = 64
SEED = 42
TEST_FRACTION = 0.15
VAL_FRACTION  = 0.15
WHEEZE_LABELS = {'Wheeze', 'Wheeze+Crackle'}

# Set to a small number (e.g. 400) for a quick trial; None uses everything.
MAX_EVENTS = None

## Step 3 — Read the data and label each event *wheeze / no-wheeze*

We keep each child's `patient_id` so the same child never appears in both the training and test sets (that would let the model cheat).

In [ ]:
import json
import pandas as pd

def read_events(json_path):
    with open(json_path) as f:
        data = json.load(f)
    events = None
    for key in ('event_annotation', 'events', 'event', 'annotation'):
        if isinstance(data.get(key), list):
            events = data[key]; break
    if events is None:
        return []
    out = []
    for ev in events:
        if not isinstance(ev, dict):
            continue
        s = ev.get('start', ev.get('start_time'))
        e = ev.get('end', ev.get('end_time'))
        t = ev.get('type', ev.get('label'))
        if s is None or e is None or t is None:
            continue
        try:
            out.append((int(float(s)), int(float(e)), str(t).strip()))
        except (TypeError, ValueError):
            continue
    return out

rows = []
for wav in sorted(WAV_DIR.glob('*.wav')):
    jp = JSON_DIR / (wav.stem + '.json')
    if not jp.exists():
        continue
    pid = wav.name.split('_')[0]
    for s, e, t in read_events(jp):
        if e <= s:
            continue
        rows.append({'wav_path': str(wav), 'start_ms': s, 'end_ms': e,
                     'patient_id': pid,
                     'label': 'wheeze' if t in WHEEZE_LABELS else 'no_wheeze'})

df = pd.DataFrame(rows)
assert not df.empty, 'Parsed 0 events - check the folder paths in Step 2.'

# Split patients (not events) into train / val / test.
patients = pd.Series(sorted(df['patient_id'].unique())).sample(frac=1.0, random_state=SEED).tolist()
n = len(patients)
n_test = max(1, round(n * TEST_FRACTION)); n_val = max(1, round(n * VAL_FRACTION))
test_p = set(patients[:n_test]); val_p = set(patients[n_test:n_test+n_val])
df['split'] = df['patient_id'].map(lambda p: 'test' if p in test_p else ('val' if p in val_p else 'train'))

if MAX_EVENTS:
    df = df.groupby('label', group_keys=False).apply(lambda g: g.sample(min(len(g), MAX_EVENTS // 2), random_state=SEED))

print('Total events:', len(df), '| patients:', df['patient_id'].nunique())
print(df['label'].value_counts())
print(df.groupby(['split', 'label']).size())

## Step 4 — Clean each clip and turn it into features

**Cleaning (Phase 3):** band-pass filter, normalise volume, fix length.  
**Features:** MFCCs (numbers, for the traditional models) and a log-mel spectrogram (a picture, for the CNN).

In [ ]:
import numpy as np, librosa, time
from scipy.signal import butter, sosfilt

CLIP_LEN = int(CLIP_SECONDS * SAMPLE_RATE)

def clean_clip(wav_path, start_ms, end_ms):
    sig, sr = librosa.load(wav_path, sr=SAMPLE_RATE, offset=start_ms/1000.0,
                           duration=(end_ms-start_ms)/1000.0)
    if len(sig) == 0:
        return np.zeros(CLIP_LEN, dtype=np.float32)
    ny = sr/2.0
    sos = butter(4, [BANDPASS_LOW/ny, min(BANDPASS_HIGH/ny, 0.99)], btype='band', output='sos')
    sig = sosfilt(sos, sig)
    peak = np.max(np.abs(sig))
    if peak > 0:
        sig = sig/peak
    if len(sig) >= CLIP_LEN:
        sig = sig[:CLIP_LEN]
    else:
        sig = np.pad(sig, (0, CLIP_LEN-len(sig)))
    return sig.astype(np.float32)

def mfcc_vec(sig):
    m = librosa.feature.mfcc(y=sig, sr=SAMPLE_RATE, n_mfcc=N_MFCC)
    return np.concatenate([m.mean(axis=1), m.std(axis=1)]).astype(np.float32)

def mel_img(sig):
    mel = librosa.feature.melspectrogram(y=sig, sr=SAMPLE_RATE, n_mels=N_MELS)
    log = librosa.power_to_db(mel, ref=np.max)
    return np.clip((log + 80.0)/80.0, 0, 1).astype(np.float32)

L = {'no_wheeze': 0, 'wheeze': 1}
X_mfcc, X_img, y, splits = [], [], [], []
t0 = time.time()
for i, (_, r) in enumerate(df.iterrows(), 1):
    try:
        c = clean_clip(r['wav_path'], r['start_ms'], r['end_ms'])
    except Exception:
        continue
    X_mfcc.append(mfcc_vec(c)); X_img.append(mel_img(c))
    y.append(L[r['label']]); splits.append(r['split'])
    if i % 500 == 0:
        print(f'  {i}/{len(df)}  ({time.time()-t0:.0f}s)')

X_mfcc = np.array(X_mfcc); X_img = np.array(X_img)
y = np.array(y); splits = np.array(splits)
print('Feature shapes:', X_mfcc.shape, X_img.shape)

In [ ]:
sel = lambda a, s: a[splits == s]
Xtr_m, Xte_m = sel(X_mfcc, 'train'), sel(X_mfcc, 'test')
Xtr_i, Xval_i, Xte_i = sel(X_img, 'train'), sel(X_img, 'val'), sel(X_img, 'test')
ytr, yval, yte = sel(y, 'train'), sel(y, 'val'), sel(y, 'test')
print('train / val / test events:', len(ytr), len(yval), len(yte))

## Step 5 — A reusable scoring helper

For a wheeze detector we care most about **sensitivity** (did we catch the wheezers?) and **specificity** (did we correctly clear the healthy kids?).

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve, f1_score

def evaluate(name, yte, pred, score):
    tn, fp, fn, tp = confusion_matrix(yte, pred, labels=[0,1]).ravel()
    sens = tp/(tp+fn) if (tp+fn) else 0
    spec = tn/(tn+fp) if (tn+fp) else 0
    auc = roc_auc_score(yte, score) if len(set(yte)) > 1 else float('nan')
    f1 = f1_score(yte, pred, zero_division=0)
    print(f'{name:15s}  sensitivity={sens:.3f}  specificity={spec:.3f}  AUC={auc:.3f}  F1={f1:.3f}')
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
    g = np.array([[tn, fp], [fn, tp]])
    ax[0].imshow(g, cmap='Blues')
    ax[0].set_xticks([0,1], ['pred no', 'pred wheeze']); ax[0].set_yticks([0,1], ['true no', 'true wheeze'])
    for a in range(2):
        for b in range(2):
            ax[0].text(b, a, g[a,b], ha='center', va='center')
    ax[0].set_title(f'{name}: confusion')
    if len(set(yte)) > 1:
        fpr, tpr, _ = roc_curve(yte, score)
        ax[1].plot(fpr, tpr, label=f'AUC={auc:.3f}'); ax[1].plot([0,1],[0,1],'--',color='grey')
        ax[1].set_xlabel('1 - specificity'); ax[1].set_ylabel('sensitivity'); ax[1].legend(); ax[1].set_title('ROC')
    plt.tight_layout(); plt.show()
    return {'sensitivity': sens, 'specificity': spec, 'auc': auc, 'f1': f1}

results = {}

## Step 6 — Train the traditional models (Random Forest, SVM, XGBoost)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=SEED, n_jobs=-1).fit(Xtr_m, ytr)
results['Random Forest'] = evaluate('Random Forest', yte, rf.predict(Xte_m), rf.predict_proba(Xte_m)[:,1])

svm = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=SEED).fit(Xtr_m, ytr)
results['SVM'] = evaluate('SVM', yte, svm.predict(Xte_m), svm.predict_proba(Xte_m)[:,1])

pos = max((ytr==0).sum()/max((ytr==1).sum(),1), 1)
xgb = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1, scale_pos_weight=pos,
                    eval_metric='logloss', random_state=SEED).fit(Xtr_m, ytr)
results['XGBoost'] = evaluate('XGBoost', yte, xgb.predict(Xte_m), xgb.predict_proba(Xte_m)[:,1])

## Step 7 — Train the CNN (medical-grade: deeper, longer training, smart early stopping)

**Key improvements for health applications:**
- **Deeper architecture** — 3 convolutional blocks instead of 2, for better feature learning
- **Batch normalization** — keeps training stable across different data distributions
- **More dropout** — prevents overfitting (crucial for small medical datasets)
- **150 epochs + early stopping** — trains longer but stops automatically when validation stops improving
- **ReduceLROnPlateau** — if the model gets stuck, reduce learning rate to find better solutions
- **Smaller batch size (16 vs 32)** — noisier gradients = better generalization to real-world patients

Target: **high sensitivity** (catch the wheezers) + **high specificity** (correctly clear the healthy kids).

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.utils.class_weight import compute_class_weight
tf.random.set_seed(SEED)

w = compute_class_weight('balanced', classes=np.array([0,1]), y=ytr)
cw = {0: w[0], 1: w[1]}

Atr, Ava, Ate = Xtr_i[..., None], Xval_i[..., None], Xte_i[..., None]

# Deeper, better-regularized architecture for medical-grade classification
cnn = models.Sequential([
    layers.Input(shape=Atr.shape[1:]),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=2),
    layers.Dropout(0.25),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=2),
    layers.Dropout(0.25),
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(pool_size=2),
    layers.Dropout(0.3),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid'),
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
cnn.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Smart callbacks: early stopping if validation plateaus, reduce LR if stuck
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)

print('Training CNN with early stopping + LR scheduling (medical-grade)...')
cnn.fit(Atr, ytr, validation_data=(Ava, yval), epochs=150, batch_size=16, 
        class_weight=cw, callbacks=[early_stop, reduce_lr], verbose=1)

sc = cnn.predict(Ate, verbose=0).ravel()
results['CNN'] = evaluate('CNN', yte, (sc >= 0.5).astype(int), sc)

## Step 8 — Final comparison

In [ ]:
print(f"{'Model':<16}{'Sens':>8}{'Spec':>8}{'AUC':>8}{'F1':>8}")
for k, m in results.items():
    print(f"{k:<16}{m['sensitivity']:>8.3f}{m['specificity']:>8.3f}{m['auc']:>8.3f}{m['f1']:>8.3f}")

print('\nRemember: this is trained on public data as a proof-of-concept.')
print('The novel step is feeding in Geetesh\'s own smartphone recordings.')